In [1]:
from pathlib import Path
import gc
import pandas as pd

RAW = Path("/scratch/sm3664/medicare-fraud/raw/part_b")
PROC = Path("/scratch/sm3664/medicare-fraud/processed/part_b")
YEARS = [2019, 2020, 2021, 2022, 2023]

def prov_path(y):
    name = f"Medicare_Physician_Other_Practitioners_by_Provider_{y}"
    return RAW / name / f"{name}.csv"

header = pd.read_csv(prov_path(2019), nrows=0).columns.tolist()
DROP = [c for c in header
        if c.startswith("Rndrng_Prvdr_") or c.startswith("Bene_Race_") or c.endswith("_Sprsn_Ind")]
ENRICH = [c for c in header if c not in DROP and c != "Rndrng_NPI"]
print(len(header), "total |", len(DROP), "dropped |", len(ENRICH), "enrichment features")
print("Dropped:", DROP)

81 total | 24 dropped | 56 enrichment features
Dropped: ['Rndrng_Prvdr_Last_Org_Name', 'Rndrng_Prvdr_First_Name', 'Rndrng_Prvdr_MI', 'Rndrng_Prvdr_Crdntls', 'Rndrng_Prvdr_Ent_Cd', 'Rndrng_Prvdr_St1', 'Rndrng_Prvdr_St2', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_RUCA_Desc', 'Rndrng_Prvdr_Cntry', 'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind', 'Drug_Sprsn_Ind', 'Med_Sprsn_Ind', 'Bene_Race_Wht_Cnt', 'Bene_Race_Black_Cnt', 'Bene_Race_API_Cnt', 'Bene_Race_Hspnc_Cnt', 'Bene_Race_NatInd_Cnt', 'Bene_Race_Othr_Cnt']


In [2]:
agg = pd.read_parquet(PROC / "part_b_agg_2019_2023.parquet", columns=["Rndrng_NPI", "year"])

null_pct, info = {}, []
for y in YEARS:
    p = pd.read_csv(prov_path(y), dtype=str)
    num = p[ENRICH].apply(lambda s: pd.to_numeric(s, errors="coerce"))
    agg_npis = set(agg.loc[agg["year"] == y, "Rndrng_NPI"])
    prov_npis = set(p["Rndrng_NPI"].dropna())
    null_pct[y] = (p[ENRICH].isna().mean() * 100).round(2)
    info.append({
        "year": y,
        "rows": len(p),
        "npi_missing": int(p["Rndrng_NPI"].isna().sum()),
        "npi_dup": int(p["Rndrng_NPI"].duplicated().sum()),
        "unparseable": int((num.isna() & p[ENRICH].notna()).sum().sum()),
        "agg_not_in_prov": len(agg_npis - prov_npis),
        "prov_not_in_agg": len(prov_npis - agg_npis),
        "drug_sprsn": p["Drug_Sprsn_Ind"].value_counts(dropna=False).to_dict(),
        "med_sprsn": p["Med_Sprsn_Ind"].value_counts(dropna=False).to_dict(),
    })
    del p, num; gc.collect()

pd.DataFrame(info)

,year,rows,npi_missing,npi_dup,unparseable,agg_not_in_prov,prov_not_in_agg,drug_sprsn,med_sprsn
0,2019,1155870,0,0,0,0,62503,"{nan: 1033475, '*': 122170, '#': 225}","{nan: 1033475, '#': 121981, '*': 414}"
1,2020,1161542,0,0,0,0,76229,"{nan: 1029854, '*': 131321, '#': 367}","{nan: 1029854, '#': 131140, '*': 548}"
2,2021,1198754,0,0,0,0,75165,"{nan: 1059977, '*': 138736, '#': 41}","{nan: 1059977, '#': 138594, '*': 183}"
3,2022,1230293,0,0,0,0,81420,"{nan: 1086928, '*': 143327, '#': 38}","{nan: 1086928, '#': 143184, '*': 181}"
4,2023,1259343,0,0,0,0,84062,"{nan: 1112903, '*': 146373, '#': 67}","{nan: 1112903, '#': 146240, '*': 200}"


In [3]:
pd.set_option("display.max_rows", 100)
nulls = pd.DataFrame(null_pct)
nulls[nulls.max(axis=1) > 0].sort_values(2019, ascending=False)

,2019,2020,2021,2022,2023
Bene_CC_BH_PTSD_V1_Pct,64.53,65.85,66.68,67.52,67.87
Bene_CC_BH_ADHD_OthCD_V1_Pct,64.45,64.75,65.96,66.51,66.86
Bene_CC_BH_PD_V1_Pct,63.59,65.29,65.68,66.20,66.04
Bene_CC_PH_Parkinson_V2_Pct,59.31,60.73,60.90,61.54,61.92
Bene_CC_BH_Schizo_OthPsy_V1_Pct,55.52,57.15,57.77,58.84,59.08
Bene_CC_BH_Bipolar_V1_Pct,51.36,54.34,53.96,55.70,56.60
Bene_Age_GT_84_Cnt,42.67,46.08,44.74,47.42,49.99
Bene_CC_BH_Alcohol_Drug_V1_Pct,41.61,44.40,43.83,45.89,47.46
Bene_CC_BH_Alz_NonAlzdem_V2_Pct,38.97,41.35,40.41,41.70,42.80
Bene_Age_LT_65_Cnt,38.27,42.42,42.30,46.47,50.09


In [4]:
FLAG_SRC = ["Drug_Sprsn_Ind", "Med_Sprsn_Ind"]
DRUG_MED = [c for c in ENRICH if c.startswith(("Drug_", "Med_"))]
DTYPES = {"Rndrng_NPI": "string", **{c: "float64" for c in ENRICH}, **{c: "string" for c in FLAG_SRC}}

def clean_prov(year):
    p = pd.read_csv(prov_path(year), usecols=["Rndrng_NPI"] + ENRICH + FLAG_SRC, dtype=DTYPES)
    p = p[p["Rndrng_NPI"].notna()]                      # paper: remove rows missing NPI
    assert not p["Rndrng_NPI"].duplicated().any()
    # validation only: Drug/Med blanks should occur only on suppressed rows
    unflagged = p["Drug_Sprsn_Ind"].isna() & p["Med_Sprsn_Ind"].isna()
    assert p.loc[unflagged, DRUG_MED].isna().sum().sum() == 0
    p = p.drop(columns=FLAG_SRC)                        # paper: drop suppression flags
    p[ENRICH] = p[ENRICH].fillna(0)                     # paper: impute missing with 0
    p.insert(1, "year", year)
    return p

In [5]:
p19 = clean_prov(2019)
a19 = pd.read_parquet(PROC / "agg" / "part_b_agg_2019.parquet")
e19 = a19.merge(p19, on=["Rndrng_NPI", "year"], how="inner", validate="one_to_one")   # paper: inner join
print("agg:", a19.shape, "| prov:", p19.shape, "| enriched:", e19.shape)
print("Rows lost from agg:", len(a19) - len(e19))
print("NaN remaining:", int(e19[ENRICH].isna().sum().sum()))

diff = e19["Tot_Srvcs"] - e19["Tot_Srvcs_sum"]
print("Tot_Srvcs < Tot_Srvcs_sum (should be 0):", int((diff < -1e-6).sum()))
print(f"Exactly equal: {(diff.abs() < 1e-6).mean():.1%}")

agg: (1093367, 34) | prov: (1155870, 58) | enriched: (1093367, 90)
Rows lost from agg: 0
NaN remaining: 0
Tot_Srvcs < Tot_Srvcs_sum (should be 0): 0
Exactly equal: 3.7%


In [6]:
ENR_DIR = PROC / "enriched"
ENR_DIR.mkdir(exist_ok=True)
del p19, a19, e19; gc.collect()

for y in YEARS:
    out = ENR_DIR / f"part_b_enriched_{y}.parquet"
    if out.exists():
        print(y, "cached, skipping")
        continue
    a = pd.read_parquet(PROC / "agg" / f"part_b_agg_{y}.parquet")
    p = clean_prov(y)
    e = a.merge(p, on=["Rndrng_NPI", "year"], how="inner", validate="one_to_one")
    print(y, e.shape, "| rows lost from agg:", len(a) - len(e))
    e.to_parquet(out, index=False)
    del a, p, e; gc.collect()

enr = pd.concat([pd.read_parquet(ENR_DIR / f"part_b_enriched_{y}.parquet") for y in YEARS], ignore_index=True)
for c in ["Rndrng_Prvdr_Type", "Rndrng_Prvdr_Ent_Cd"]:
    enr[c] = enr[c].astype("category")
assert not enr.duplicated(["Rndrng_NPI", "year"]).any()
print(enr.shape)
enr.to_parquet(PROC / "part_b_enriched_2019_2023.parquet", index=False)

2019 (1093367, 90) | rows lost from agg: 0
2020 (1085313, 90) | rows lost from agg: 0
2021 (1123589, 90) | rows lost from agg: 0
2022 (1148873, 90) | rows lost from agg: 0
2023 (1175281, 90) | rows lost from agg: 0
(5626423, 90)


In [7]:
import pandas as pd
enr = pd.read_parquet("/scratch/sm3664/medicare-fraud/processed/part_b/part_b_enriched_2019_2023.parquet")

print(enr.shape)                                  # expect (5626423, 90)
print(enr.groupby("year").size())                 # rows per year
print("Unique NPIs:", enr["Rndrng_NPI"].nunique())
print("Any NaN:", int(enr.isna().sum().sum()))    # expect 0
print(list(enr.columns))                          # all 90 column names
enr.head()

(5626423, 90)
year
2019    1093367
2020    1085313
2021    1123589
2022    1148873
2023    1175281
dtype: int64
Unique NPIs: 1493778
Any NaN: 0
['Rndrng_NPI', 'year', 'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_Ent_Cd', 'Tot_Benes_min', 'Tot_Benes_max', 'Tot_Benes_median', 'Tot_Benes_mean', 'Tot_Benes_sum', 'Tot_Benes_std', 'Tot_Srvcs_min', 'Tot_Srvcs_max', 'Tot_Srvcs_median', 'Tot_Srvcs_mean', 'Tot_Srvcs_sum', 'Tot_Srvcs_std', 'Tot_Bene_Day_Srvcs_min', 'Tot_Bene_Day_Srvcs_max', 'Tot_Bene_Day_Srvcs_median', 'Tot_Bene_Day_Srvcs_mean', 'Tot_Bene_Day_Srvcs_sum', 'Tot_Bene_Day_Srvcs_std', 'Avg_Sbmtd_Chrg_min', 'Avg_Sbmtd_Chrg_max', 'Avg_Sbmtd_Chrg_median', 'Avg_Sbmtd_Chrg_mean', 'Avg_Sbmtd_Chrg_sum', 'Avg_Sbmtd_Chrg_std', 'Avg_Mdcr_Pymt_Amt_min', 'Avg_Mdcr_Pymt_Amt_max', 'Avg_Mdcr_Pymt_Amt_median', 'Avg_Mdcr_Pymt_Amt_mean', 'Avg_Mdcr_Pymt_Amt_sum', 'Avg_Mdcr_Pymt_Amt_std', 'Tot_HCPCS_Cds', 'Tot_Benes', 'Tot_Srvcs', 'Tot_Sbmtd_Chrg', 'Tot_Mdcr_Alowd_Amt', 'Tot_Mdcr_Pymt_Amt', 'Tot_Mdcr_Stdzd_Amt', '

,Rndrng_NPI,year,Rndrng_Prvdr_Type,Rndrng_Prvdr_Ent_Cd,Tot_Benes_min,Tot_Benes_max,Tot_Benes_median,Tot_Benes_mean,Tot_Benes_sum,Tot_Benes_std,...,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre
0,1003000126,2019,Internal Medicine,I,21.0,306.0,40.0,107.666667,969.0,106.070731,...,54.0,46.0,75.0,75.0,50.0,13.0,3.0,59.0,27.0,2.5917
1,1003000134,2019,Pathology,I,16.0,3414.0,92.5,397.750000,4773.0,955.578561,...,21.0,11.0,72.0,64.0,20.0,15.0,3.0,42.0,6.0,1.1246
2,1003000142,2019,Anesthesiology,I,11.0,106.0,29.5,39.388889,709.0,30.076645,...,34.0,21.0,65.0,75.0,27.0,10.0,0.0,75.0,10.0,1.6146
3,1003000423,2019,Obstetrics & Gynecology,I,16.0,43.0,23.0,27.333333,82.0,14.011900,...,19.0,0.0,58.0,53.0,0.0,15.0,0.0,43.0,0.0,0.9065
4,1003000480,2019,General Surgery,I,17.0,35.0,21.0,24.333333,73.0,9.451631,...,38.0,18.0,49.0,72.0,20.0,17.0,0.0,55.0,0.0,1.7191
